'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''



### The main purpose of this notebook is to prepare training data for graph neural network/link prediction tasks, specifically constructing unconnected node pairs and their time-varying solutions.Build(unconnected pairs) [description] (solution)(2016→2019→2022)Process

## prepare unconnected pairs and its solution year_delta=3

In [ ]:
import os
import sys
import json
import pickle
import gzip
from datetime import datetime, date
import random, time
import numpy as np
from scipy import sparse
import networkx as nx
from collections import defaultdict,Counter
import itertools
import copy
from itertools import combinations
import pandas as pd

### read full graph
#### ['v1', 'v2', 'time', 'ct', 'c2025','c2024;,'c2023', 'c2022', 'c2021', 'c2020', 'c2019', 'c2018', 'c2017', 'c2016', 'c2015', 'c2014', 'c2013', 'c2012'])

Read complete dynamic graph data including v1, v2, timestamp and yearly citation counts from parquet file (full_dynamic_graph.parquet).v1v2 [description] ,parquetRead complete dynamic graph data including v1, v2, timestamp and yearly citation counts from parquet file (full_dynamic_graph.parquet).(full_dynamic_graph.parquet)
Data volume: 1,170,455 edges

In [ ]:
time_start = time.time()

graph_folder="./data/data_concept_graph"
graph_file=os.path.join(graph_folder,"full_dynamic_graph.parquet")

full_graph = pd.read_parquet(graph_file)
print(f"Done, read full_graph: {len(full_graph)}; elapsed_time: {time.time() - time_start} seconds")

Checkv1 < v2Sort [description] Store

In [ ]:
time_start=time.time()
is_smaller = np.all(full_graph['v1'] < full_graph['v2'])
print(f"is_smaller: {is_smaller}; elapsed_time: {time.time()-time_start}\n")

### get the unconnected-pairs years_delta=3

In [ ]:
# Code comment
NUM_OF_VERTICES=38739 ## number of concepts

vertex_degree_cutoff=1
min_edges=1
years_delta=3

day_origin = date(1990,1,1)
day_2016 = (date(2016, 12, 31)- day_origin).days
day_2019 = (date(2019, 12, 31)- day_origin).days
day_2022 = (date(2022, 12, 31)- day_origin).days
print(f"day_2016: {day_2016}; day_2019: {day_2019}; day_2022: {day_2022}\n")

#### get full_graph up to 2016,2019,2022

201620192022

In [ ]:
print(f"full_dynamic_graph: {len(full_graph)}")
time_start = time.time()
full_graph_2016=full_graph[full_graph['time']<=day_2016]
print(f"full_graph_2016: {len(full_graph_2016)}; elapsed_time: {time.time()-time_start}")


time_start = time.time()
full_graph_2019=full_graph[full_graph['time']<=day_2019]
print(f"full_graph_2019: {len(full_graph_2019)}; elapsed_time: {time.time()-time_start}")


time_start = time.time()
full_graph_2022=full_graph[full_graph['time']<=day_2022]
print(f"full_graph_2022: {len(full_graph_2022)}; elapsed_time: {time.time()-time_start}")

#### get all the vertex pairs

Extract(v1, v2)Deduplication

In [ ]:
time_start=time.time()
pairs_2016 = full_graph_2016[['v1', 'v2']].drop_duplicates()
print(f"pairs_2016: {len(pairs_2016)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
pairs_2019 = full_graph_2019[['v1', 'v2']].drop_duplicates()
print(f"pairs_2019: {len(pairs_2019)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
pairs_2022 = full_graph_2022[['v1', 'v2']].drop_duplicates()
print(f"pairs_2022: {len(pairs_2022)}; elapsed_time: {time.time()-time_start}")

#### get all-combine-pairs while degree >= vertex_degree_cutoff

Calculate items [description] ≥1

In [ ]:
time_start=time.time()
# Flatten the array and count the frequency of each node (this gives the degree of each node)
all_nodes_2016, degrees_2016 = np.unique(pairs_2016.values.flatten(), return_counts=True)

# Create a mask for nodes with a degree greater than the cutoff
large_degree_mask = degrees_2016 >= vertex_degree_cutoff
# Get the nodes with a degree greater than the cutoff
vertex_large_degs_2016 = all_nodes_2016[large_degree_mask]
print(f"vertex_used_2016: {len(vertex_large_degs_2016)}; elapsed_time: {time.time()-time_start}")


time_start=time.time()
all_nodes_2019, degrees_2019 = np.unique(pairs_2019.values.flatten(), return_counts=True)
large_degree_mask = degrees_2019 >= vertex_degree_cutoff
vertex_large_degs_2019 = all_nodes_2019[large_degree_mask]
print(f"vertex_used_2019: {len(vertex_large_degs_2019)}; elapsed_time: {time.time()-time_start}")


time_start=time.time()
all_nodes_2022, degrees_2022 = np.unique(pairs_2022.values.flatten(), return_counts=True)
large_degree_mask = degrees_2022 >= vertex_degree_cutoff
vertex_large_degs_2022 = all_nodes_2022[large_degree_mask]
print(f"vertex_used_2022: {len(vertex_large_degs_2022)}; elapsed_time: {time.time()-time_start}")

#### get all the combination of the used vertex

 [description] Generate [description] C(n,2)

Data volume20161.8620221.97

In [ ]:
 
time_start=time.time()
n = len(vertex_large_degs_2016)
c, r = np.triu_indices(n, k=1)  # Gets the upper triangle indices excluding the diagonal
combine_pairs_2016 = np.column_stack((vertex_large_degs_2016[c], vertex_large_degs_2016[r]))
print(f"all combine_pairs_2016: {len(combine_pairs_2016)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
n = len(vertex_large_degs_2019)
c, r = np.triu_indices(n, k=1)  # Gets the upper triangle indices excluding the diagonal
combine_pairs_2019 = np.column_stack((vertex_large_degs_2019[c], vertex_large_degs_2019[r]))
print(f"all combine_pairs_2019: {len(combine_pairs_2019)}; elapsed_time: {time.time()-time_start}")


time_start=time.time()
n = len(vertex_large_degs_2022)
c, r = np.triu_indices(n, k=1)  # Gets the upper triangle indices excluding the diagonal
combine_pairs_2022 = np.column_stack((vertex_large_degs_2022[c], vertex_large_degs_2022[r]))
print(f"all combine_pairs_2022: {len(combine_pairs_2022)}; elapsed_time: {time.time()-time_start}")

GenerateSave aspickle

In [ ]:
import pickle

# Save combine_pairs_2016
with open("combine_pairs_2016.pkl", "wb") as f:
    pickle.dump(combine_pairs_2016, f, protocol=4)

# Save combine_pairs_2019
with open("combine_pairs_2019.pkl", "wb") as f:
    pickle.dump(combine_pairs_2019, f, protocol=4)

# Save combine_pairs_2022
with open("combine_pairs_2022.pkl", "wb") as f:
    pickle.dump(combine_pairs_2022, f, protocol=4)

print("✅ combine_pairs_2016 / 2019 / 2022 ...SaveDone")


pickleLoad

In [ ]:
import pickle

with open("combine_pairs_2016.pkl", "rb") as f:
    combine_pairs_2016 = pickle.load(f)

with open("combine_pairs_2019.pkl", "rb") as f:
    combine_pairs_2019 = pickle.load(f)

with open("combine_pairs_2022.pkl", "rb") as f:
    combine_pairs_2022 = pickle.load(f)

print("✅ combine_pairs_2016 / 2019 / 2022 ...SuccessfullyLoad")


numpypandas DataFrame

In [ ]:
# Convert numpy arrays to pandas DataFrames
time_start=time.time()
all_combine_pairs_2016 = pd.DataFrame(combine_pairs_2016, columns=['v1', 'v2'])
print(f"Convert combine_pairs_2016: {len(all_combine_pairs_2016)}, elapsed_time: {time.time()-time_start}")

time_start=time.time()
all_combine_pairs_2019 = pd.DataFrame(combine_pairs_2019, columns=['v1', 'v2'])
print(f"Convert combine_pairs_2019: {len(all_combine_pairs_2019)}, elapsed_time: {time.time()-time_start}")

time_start=time.time()
all_combine_pairs_2022 = pd.DataFrame(combine_pairs_2022, columns=['v1', 'v2'])
print(f"Convert combine_pairs_2022: {len(all_combine_pairs_2022)}, elapsed_time: {time.time()-time_start}")

### prepare unconnected_pairs

#### unconnected pairs in 2016 2016
 [description] (1.86) [description] (85)

1.85 items2016

Process

In [ ]:
import pandas as pd
import time
import gc
import os

time_start = time.time()

print("Build......")
# Code comment
connected_pairs_set = set(zip(pairs_2016['v1'], pairs_2016['v2']))
print(f"...Size: {len(connected_pairs_set):,}")

  # Comment
BATCH_SIZE = 2_000_000
total_rows = len(all_combine_pairs_2016)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\nStart...Process... rows...: {total_rows:,}... {num_batches} ...Process")

# Code comment
all_unconnected_chunks = []
total_unconnected = 0

for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    
    print(f"▶ Process... {i+1}/{num_batches} ...:  rows {batch_start:,} - {batch_end:,}")
    
      # Comment
    batch_df = all_combine_pairs_2016.iloc[batch_start:batch_end].copy()
    
    # Code comment
    is_connected = pd.Series(
        [(v1, v2) in connected_pairs_set for v1, v2 in zip(batch_df['v1'], batch_df['v2'])],
        index=batch_df.index
    )
    
    # Code comment
    unconnected_batch = batch_df[~is_connected]
    
    # Code comment
    if len(unconnected_batch) > 0:
          # Comment
        unconnected_batch = unconnected_batch.reset_index(drop=True)
        all_unconnected_chunks.append(unconnected_batch)
        total_unconnected += len(unconnected_batch)
        print(f"  Found...: {len(unconnected_batch):,}  rows (Cumulative: {total_unconnected:,})")
    else:
        print(f"  ...")
    
      # Comment
    del batch_df, is_connected, unconnected_batch
    gc.collect()

print(f"\n...Processing complete...Start mergingResult...")
print(f"...: {total_unconnected:,}")

# Code comment
if total_unconnected > 0:
    print("...Merge......")
    
    # Code comment
    if len(all_unconnected_chunks) > 10:
        print(f"... {len(all_unconnected_chunks)}  items...Merge...")
        
        # Code comment
        merged_chunks = []
        for j in range(0, len(all_unconnected_chunks), 10):
            end_idx = min(j + 10, len(all_unconnected_chunks))
            batch_to_merge = all_unconnected_chunks[j:end_idx]
            
            print(f"  Merge... {j//10 + 1}: {len(batch_to_merge)}  items...")
            merged_batch = pd.concat(batch_to_merge, ignore_index=True)
            merged_chunks.append(merged_batch)
            
              # Comment
            del batch_to_merge
            gc.collect()
        
        # Code comment
        print("... rows...Merge...")
        unconnected_pairs_2016 = pd.concat(merged_chunks, ignore_index=True)
        
    else:
        # Code comment
        unconnected_pairs_2016 = pd.concat(all_unconnected_chunks, ignore_index=True)
    
    print(f"✅ Merge complete! ... rows...: {len(unconnected_pairs_2016):,}")
    
      # Comment
    output_file = "unconnected_pairs_2016.parquet"
    print(f"...Save to {output_file}...")
    
    # Code comment
    unconnected_pairs_2016.to_parquet(output_file, compression='snappy')
    print(f"✅ ...Save to {output_file}")
    
      # Comment
    print(f"\nStatistics...:")
    print(f"  ... rows...: {len(unconnected_pairs_2016):,}")
    print(f"   columns...: {list(unconnected_pairs_2016.columns)}")
    print(f"  ... rows...:")
    print(unconnected_pairs_2016.head())
    
else:
    print("...Found...")
    unconnected_pairs_2016 = pd.DataFrame(columns=['v1', 'v2'])

print(f"\n✅ Processing complete!")
print(f"Total time: {time.time()-time_start:.2f} s")

  # Comment
del all_unconnected_chunks
gc.collect()

ResultSave asparquet

In [ ]:
import pandas as pd

# Code comment
unconnected_pairs_2016.to_parquet("unconnected_pairs_2016.parquet", compression='snappy')

  # Comment
del unconnected_pairs_2016
import gc; gc.collect()

print("✅ ...Save as Parquet ...")


In [ ]:
import pandas as pd
unconnected_pairs_2016 = pd.read_parquet("unconnected_pairs_2016.parquet")


#### check unconnected pairs in 2016 for 2019 (unconnected+citation and connected+citation)
Check20162019

In [ ]:
### in unconnected_pair_2016 but not in pairs_2019
### unconnected pairs keep unconnected in 2019
import time
import pandas as pd
time_start=time.time()

unconnected_pair_2016_2019 = pd.merge(unconnected_pairs_2016, pairs_2019, on=['v1', 'v2'], how='outer', indicator=True)
unconnected_pair_2016_2019 = unconnected_pair_2016_2019[unconnected_pair_2016_2019['_merge'] == 'left_only']
unconnected_pair_2016_2019 = unconnected_pair_2016_2019.drop(columns=['_merge'])

print(f"unconnected_pair_2016_2019: {len(unconnected_pair_2016_2019)}; elapsed_time: {time.time()-time_start}")

### in unconnected_pair_2016 but also in pairs_2019
### unconnected pairs becomes connected in 2019
time_start=time.time()
connected_pair_2016_2019 = pd.merge(pairs_2019,unconnected_pairs_2016, on=['v1', 'v2'], how='inner', indicator=True)
connected_pair_2016_2019 = connected_pair_2016_2019[connected_pair_2016_2019['_merge'] == 'both']
connected_pair_2016_2019 = connected_pair_2016_2019.drop(columns=['_merge'])

print(f"connected_pair_2016_2019: {len(connected_pair_2016_2019)}; elapsed_time: {time.time()-time_start}\n")
print(f"train 2016-2019: total- {len(unconnected_pairs_2016)}; connected-- {len(connected_pair_2016_2019)}; unconnected--{len(unconnected_pair_2016_2019)}")

1.85(50/)2019 rowsmerge

In [ ]:
##################################################################################################
# Code comment
##################################################################################################
import gc
BATCH_SIZE = 500000  # Code comment
output_folder = "batch_results"
os.makedirs(output_folder, exist_ok=True)

time_start = time.time()
total_rows = len(unconnected_pairs_2016)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"... {total_rows}  rows... {num_batches} ...Process...\n")

for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    print(f"▶ ...Process... {i + 1}/{num_batches} ...:  rows {batch_start} - {batch_end}")

      # Comment
    batch_df = unconnected_pairs_2016.iloc[batch_start:batch_end].copy()

      # Comment
    batch_unconnected = pd.merge(batch_df, pairs_2019, on=['v1', 'v2'], how='outer', indicator=True)
    batch_unconnected = batch_unconnected[batch_unconnected['_merge'] == 'left_only'].drop(columns=['_merge'])

      # Comment
    batch_connected = pd.merge(pairs_2019, batch_df, on=['v1', 'v2'], how='inner', indicator=True)
    batch_connected = batch_connected[batch_connected['_merge'] == 'both'].drop(columns=['_merge'])

    # Save
    unconnected_path = os.path.join(output_folder, f"unconnected_pair_2016_2019_part{i+1}.parquet")
    connected_path = os.path.join(output_folder, f"connected_pair_2016_2019_part{i+1}.parquet")

    batch_unconnected.to_parquet(unconnected_path, compression="snappy")
    batch_connected.to_parquet(connected_path, compression="snappy")

    print(f"  ...Save... {len(batch_unconnected)}  rows... {len(batch_connected)}  rows")

      # Comment
    del batch_df, batch_unconnected, batch_connected
    gc.collect()

print(f"\n✅ ...Processing complete...Total time {time.time() - time_start:.2f} s")

##################################################################################################
  # Comment
##################################################################################################

def merge_all_parts(prefix, output_name):
    print(f"\n...Merge... {prefix} ......")
    all_files = sorted([f for f in os.listdir(output_folder) if f.startswith(prefix)])
    dfs = []
    for f in all_files:
        df = pd.read_parquet(os.path.join(output_folder, f))
        dfs.append(df)
    merged = pd.concat(dfs, ignore_index=True)
    merged.to_parquet(output_name, compression="snappy")
    print(f"✅ ...MergeSave as {output_name}... rows...: {len(merged)}")
    del dfs, merged
    gc.collect()

# Code comment
# merge_all_parts("connected_pair_2016_2019_part", "connected_pair_2016_2019.parquet")
# merge_all_parts("unconnected_pair_2016_2019_part", "unconnected_pair_2016_2019.parquet")


### Merge connected_pair_2016_2019.parquetunconnected_pair_2016_2019.parquet

In [ ]:
import pandas as pd
import os
import gc
import time

def merge_large_files_incrementally(input_folder, file_prefix, output_file, batch_size=5):
    """
    ...Merge [description] 
    
    Parameters:
    -----------
    input_folder : str - ...
    file_prefix : str - ... "connected_pair_2016_2019_part"...
    output_file : str - ...Output...
    batch_size : int - ...Merge...File count [description] 5...
    """
    print(f"\nStart...Merge...: {file_prefix}")
    print(f"Output...: {output_file}")
    print(f"...Merge {batch_size}  files\n")
    
    # Code comment
    all_files = sorted([f for f in os.listdir(input_folder) 
                        if f.startswith(file_prefix) and f.endswith('.parquet')])
    
    if not all_files:
        print(f"Error: ... {input_folder} ...Found... {file_prefix} ...parquet...")
        return
    
    total_files = len(all_files)
    print(f"Found {total_files}  items...")
    
      # Comment
    temp_folder = os.path.join(input_folder, "temp_merge")
    os.makedirs(temp_folder, exist_ok=True)
    
    # Code comment
    intermediate_files = []
    num_batches = (total_files + batch_size - 1) // batch_size
    
    print(f"\n...Merge... {num_batches}  items...")
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, total_files)
        batch_files = all_files[start_idx:end_idx]
        
        print(f"\n▶ Merge... {batch_idx + 1}/{num_batches}: Process {len(batch_files)}  files")
        
          # Comment
        dfs = []
        batch_rows = 0
        for i, file_name in enumerate(batch_files):
            file_path = os.path.join(input_folder, file_name)
            print(f"  Read... {i+1}/{len(batch_files)}: {file_name}")
            
            try:
                df = pd.read_parquet(file_path)
                rows = len(df)
                batch_rows += rows
                dfs.append(df)
                print(f"     rows...: {rows:,}")
            except Exception as e:
                print(f"    ReadFailed: {e}")
                continue
        
        if not dfs:
            print("  ...SuccessfullyRead...Skip")
            continue
        
          # Comment
        print(f"  Merge... {len(dfs)}  files... rows...: {batch_rows:,}")
        merged_batch = pd.concat(dfs, ignore_index=True)
        
          # Comment
        temp_file = os.path.join(temp_folder, f"temp_batch_{batch_idx+1}.parquet")
        merged_batch.to_parquet(temp_file, compression='snappy')
        intermediate_files.append(temp_file)
        
        print(f"  ...Save...: {temp_file}")
        print(f"  ... rows...: {len(merged_batch):,}")
        
          # Comment
        del dfs, merged_batch
        gc.collect()
    
    # Code comment
    print(f"\n...Merge {len(intermediate_files)}  items...")
    
    if len(intermediate_files) == 1:
        # Code comment
        print("... items [description] Result")
        os.rename(intermediate_files[0], output_file)
        final_rows = pd.read_parquet(output_file).shape[0]
    else:
        # Code comment
        final_dfs = []
        total_final_rows = 0
        
        # Code comment
        if len(intermediate_files) > batch_size:
            print(" [description] Merge...")
            
            # Code comment
            second_level_files = []
            for i in range(0, len(intermediate_files), batch_size):
                batch = intermediate_files[i:i+batch_size]
                print(f"  Merge... {i//batch_size + 1}: {len(batch)}  files")
                
                batch_dfs = []
                for f in batch:
                    df = pd.read_parquet(f)
                    batch_dfs.append(df)
                    os.remove(f)    # Comment
                
                merged = pd.concat(batch_dfs, ignore_index=True)
                temp_second = os.path.join(temp_folder, f"temp_second_{i//batch_size+1}.parquet")
                merged.to_parquet(temp_second, compression='snappy')
                second_level_files.append(temp_second)
                
                del batch_dfs, merged
                gc.collect()
            
            intermediate_files = second_level_files
        
        # Code comment
        print(f"...Merge {len(intermediate_files)}  files...")
        for f in intermediate_files:
            df = pd.read_parquet(f)
            final_dfs.append(df)
            total_final_rows += len(df)
            os.remove(f)  # DeleteTemp file
        
          # Comment
        print(f"Merge...Result... rows...: {total_final_rows:,}")
        final_result = pd.concat(final_dfs, ignore_index=True)
        final_result.to_parquet(output_file, compression='snappy')
        final_rows = len(final_result)
        
        del final_dfs, final_result
        gc.collect()
    
      # Comment
    try:
        os.rmdir(temp_folder)
    except:
        print(f"Temp file... {temp_folder} ...Clean")
    
    print(f"\n✅ Merge complete...")
    print(f"...: {output_file}")
    print(f"... rows...: {final_rows:,}")
    print(f"...Size: {os.path.getsize(output_file) / (1024**3):.2f} GB")

# Code comment
if __name__ == "__main__":
    input_folder = "batch_results"  # Code comment
    
    # Code comment
      # Comment
      # Comment
      # Comment
    BATCH_SIZE = 5  # Code comment
    
      # Comment
    merge_large_files_incrementally(
        input_folder=input_folder,
        file_prefix="connected_pair_2016_2019_part",
        output_file="connected_pair_2016_2019.parquet",
        batch_size=BATCH_SIZE
    )
    
      # Comment
    gc.collect()
    
      # Comment
    # Code comment
    merge_large_files_incrementally(
        input_folder=input_folder,
        file_prefix="unconnected_pair_2016_2019_part", 
        output_file="unconnected_pair_2016_2019.parquet",
        batch_size=2  # Code comment
    )

In [ ]:
import pandas as pd

connected_pair_2016_2019 = pd.read_parquet("connected_pair_2016_2019.parquet")
unconnected_pair_2016_2019 = pd.read_parquet("unconnected_pair_2016_2019.parquet")

print(f"connected_pair_2016_2019: {len(connected_pair_2016_2019)}, unconnected_pair_2016_2019: {len(unconnected_pair_2016_2019)}")


#### unconnected pairs in 2019
20162019(1.93)(92)

1.93 items2019

ProcessSave

In [ ]:
import time
import pandas as pd
import gc
import os

time_start = time.time()

  # Comment
BATCH_SIZE = 250000
total_rows = len(all_combine_pairs_2019)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"... rows...: {total_rows:,}... {num_batches} ...Process")

  # Comment
temp_folder = "temp_unconnected_2019"
os.makedirs(temp_folder, exist_ok=True)

# Code comment
for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    
    print(f"\n▶ Process... {i+1}/{num_batches} ...:  rows {batch_start:,} - {batch_end:,}")
    
      # Comment
    batch_df = all_combine_pairs_2019.iloc[batch_start:batch_end].copy()
    
    # Code comment
    batch_merged = pd.merge(batch_df, pairs_2019, on=['v1', 'v2'], how='outer', indicator=True)
    batch_unconnected = batch_merged[batch_merged['_merge'] == 'left_only'].drop(columns=['_merge'])
    
      # Comment
    temp_file = os.path.join(temp_folder, f"batch_{i+1}.parquet")
    batch_unconnected.to_parquet(temp_file, compression='snappy')
    
    print(f"  ...Found...: {len(batch_unconnected):,}  rows")
    
      # Comment
    del batch_df, batch_merged, batch_unconnected
    gc.collect()

print(f"\n✅ ...Processing complete...Start mergingResult...")

  # Comment
all_files = sorted([f for f in os.listdir(temp_folder) if f.startswith('batch_')])
dfs = []
total_unconnected = 0

for f in all_files:
    df = pd.read_parquet(os.path.join(temp_folder, f))
    total_unconnected += len(df)
    dfs.append(df)
    print(f"Read {f}: {len(df):,}  rows (Cumulative: {total_unconnected:,})")

# Code comment
unconnected_pairs_2019 = pd.concat(dfs, ignore_index=True)
print(f"\n... unconnected_pairs_2019: {len(unconnected_pairs_2019):,}  rows")

  # Comment
unconnected_pairs_2019.to_parquet("unconnected_pairs_2019.parquet", compression='snappy')
print(f"...Save to unconnected_pairs_2019.parquet")

# CleanTemp file
import shutil
shutil.rmtree(temp_folder)

print(f"\nTotal time: {time.time()-time_start:.2f} s")

In [ ]:
import pandas as pd

# # Code comment
unconnected_pairs_2019.to_parquet("unconnected_pairs_2019.parquet", compression='snappy')

  # Comment
#del unconnected_pairs_2016
import gc; gc.collect()

print("✅ ...Save as Parquet ...")

In [ ]:
import pandas as pd
unconnected_pairs_2019 = pd.read_parquet("unconnected_pairs_2019.parquet")

#### check unconnected pairs in 2019 for 2022 (unconnected+citation and connected+citation)

In [ ]:
# time_start=time.time()
# unconnected_pair_2019_2022 = pd.merge(unconnected_pairs_2019, pairs_2022, on=['v1', 'v2'], how='outer', indicator=True)
# unconnected_pair_2019_2022 = unconnected_pair_2019_2022[unconnected_pair_2019_2022['_merge'] == 'left_only']
# unconnected_pair_2019_2022 = unconnected_pair_2019_2022.drop(columns=['_merge'])
# print(f"unconnected_pair_2019_2022: {len(unconnected_pair_2019_2022)}; elapsed_time: {time.time()-time_start}")


# time_start=time.time()
# connected_pair_2019_2022 = pd.merge(pairs_2022, unconnected_pairs_2019, on=['v1', 'v2'], how='inner', indicator=True)
# connected_pair_2019_2022 = connected_pair_2019_2022[connected_pair_2019_2022['_merge'] == 'both']
# connected_pair_2019_2022 = connected_pair_2019_2022.drop(columns=['_merge'])
# print(f"connected_pair_2019_2022, {len(connected_pair_2019_2022)}; elapsed_time: {time.time()-time_start}")
# print(f"eval 2019-2022: total- {len(unconnected_pairs_2019)}; connected-- {len(connected_pair_2019_2022)}; unconnected--{len(unconnected_pair_2019_2022)}")

1.93(200/)2022 rowsmerge

Result

1.93

1.9

In [ ]:
##################################################################################################
# Code comment
##################################################################################################
import gc
BATCH_SIZE = 2_000_000  # Code comment
output_folder = "batch_results_2019_2022"
os.makedirs(output_folder, exist_ok=True)

time_start = time.time()
total_rows = len(unconnected_pairs_2019)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\n... {total_rows}  rows... {num_batches} ...Process (2019→2022)...\n")

for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    print(f"▶ ...Process... {i + 1}/{num_batches} ...:  rows {batch_start} - {batch_end}")

    # Code comment
    batch_df = unconnected_pairs_2019.iloc[batch_start:batch_end].copy()

      # Comment
    batch_unconnected = pd.merge(batch_df, pairs_2022, on=['v1', 'v2'], how='outer', indicator=True)
    batch_unconnected = batch_unconnected[batch_unconnected['_merge'] == 'left_only'].drop(columns=['_merge'])

      # Comment
    batch_connected = pd.merge(pairs_2022, batch_df, on=['v1', 'v2'], how='inner', indicator=True)
    batch_connected = batch_connected[batch_connected['_merge'] == 'both'].drop(columns=['_merge'])

    # Save parquet
    unconnected_path = os.path.join(output_folder, f"unconnected_pair_2019_2022_part{i+1}.parquet")
    connected_path = os.path.join(output_folder, f"connected_pair_2019_2022_part{i+1}.parquet")

    batch_unconnected.to_parquet(unconnected_path, compression="snappy")
    batch_connected.to_parquet(connected_path, compression="snappy")

    print(f"  ...Save... {len(batch_unconnected)}  rows... {len(batch_connected)}  rows")

      # Comment
    del batch_df, batch_unconnected, batch_connected
    gc.collect()

print(f"\n✅ ...Processing complete...Total time {time.time() - time_start:.2f} s")

##################################################################################################
  # Comment
##################################################################################################

def merge_all_parts(prefix, output_name, folder):
    print(f"\n...Merge... {prefix} ......")
    all_files = sorted([f for f in os.listdir(folder) if f.startswith(prefix)])
    dfs = []
    for f in all_files:
        df = pd.read_parquet(os.path.join(folder, f))
        dfs.append(df)
    merged = pd.concat(dfs, ignore_index=True)
    merged.to_parquet(output_name, compression="snappy")
    print(f"✅ ...MergeSave as {output_name}... rows...: {len(merged)}")
    del dfs, merged
    gc.collect()

In [ ]:
##################################################################################################
# Code comment
# Code comment
##################################################################################################
import pandas as pd
import os
import gc
import time
import glob

def merge_all_parts_ultra_safe(prefix, output_name, folder):
    """
    ...Merge... filesProcess...
    
    Parameters:
    - prefix: ... "connected_pair_2019_2022_part"...
    - output_name: Output... "connected_pair_2019_2022.parquet"...
    - folder:  [description]  "batch_results_2019_2022"...
    """
    time_start = time.time()
    
    # Code comment
    pattern = os.path.join(folder, f"{prefix}*.parquet")
    all_files = sorted(glob.glob(pattern))
    
    print(f"\n...Merge {prefix} ......")
    print(f"Found {len(all_files)}  items...")
    
    if not all_files:
        print("...Found...")
        return
    
      # Comment
    print(f"Processing file 1/{len(all_files)}: {os.path.basename(all_files[0])}")
    df_first = pd.read_parquet(all_files[0])
    df_first.to_parquet(output_name, compression='snappy', index=False)
    total_rows = len(df_first)
    print(f"  Create...Write {len(df_first):,}  rows")
    
    # Code comment
    del df_first
    gc.collect()
    
    # Code comment
    for i, file_path in enumerate(all_files[1:], start=2):
        print(f"Processing file {i}/{len(all_files)}: {os.path.basename(file_path)}")
        
          # Comment
        df = pd.read_parquet(file_path)
        
          # Comment
        df_existing = pd.read_parquet(output_name)
        df_combined = pd.concat([df_existing, df], ignore_index=True)
        df_combined.to_parquet(output_name, compression='snappy', index=False)
        
        total_rows = len(df_combined)
        print(f"  ...Write {len(df):,}  rows...Cumulative: {total_rows:,}  rows")
        
        # Code comment
        del df, df_existing, df_combined
        gc.collect()
    
    print(f"\n✅ Merge complete... rows...: {total_rows:,}")
    print(f"  Save as: {output_name}")
    print(f"  ...: {time.time()-time_start:.2f} s")

##################################################################################################
# Code comment
##################################################################################################
def merge_all_parts_efficient(prefix, output_name, folder, temp_suffix='.tmp'):
    """
    ...Merge...Temp file...Read... files
    
    Parameters:
    - prefix: ...
    - output_name: Output...
    - folder: ...
    - temp_suffix: Temp file...
    """
    time_start = time.time()
    
    # Code comment
    pattern = os.path.join(folder, f"{prefix}*.parquet")
    all_files = sorted(glob.glob(pattern))
    
    print(f"\n...Merge {prefix} ......")
    print(f"Found {len(all_files)}  items...")
    
    if not all_files:
        print("...Found...")
        return
    
      # Comment
    temp_file = output_name + temp_suffix
    
      # Comment
    print(f"Processing file 1/{len(all_files)}: {os.path.basename(all_files[0])}")
    df_first = pd.read_parquet(all_files[0])
    df_first.to_parquet(temp_file, compression='snappy', index=False)
    total_rows = len(df_first)
    print(f"  ...CreateDone...{len(df_first):,}  rows")
    
    del df_first
    gc.collect()
    
    # Code comment
    for i, file_path in enumerate(all_files[1:], start=2):
        print(f"Processing file {i}/{len(all_files)}: {os.path.basename(file_path)}")
        
        # ReadCurrent file
        df_current = pd.read_parquet(file_path)
        
        # ReadTemp file
        df_temp = pd.read_parquet(temp_file)
        
        # Merge
        df_combined = pd.concat([df_temp, df_current], ignore_index=True)
        df_combined.to_parquet(temp_file, compression='snappy', index=False)
        
        total_rows = len(df_combined)
        print(f"  ...Done...Cumulative: {total_rows:,}  rows")
        
        # Code comment
        del df_current, df_temp, df_combined
        gc.collect()
    
      # Comment
    if os.path.exists(output_name):
        os.remove(output_name)
    os.rename(temp_file, output_name)
    
    print(f"\n✅ Merge complete... rows...: {total_rows:,}")
    print(f"  Save as: {output_name}")
    print(f"  ...: {time.time()-time_start:.2f} s")

##################################################################################################
# Code comment
##################################################################################################
def merge_parquet_files_cli(prefix, output_name, folder):
    """
    ... pyarrow ... pandas ... concat ...Merge...
    ... rows... parquet-tools
    """
    import subprocess
    
    print(f"... parquet-tools ... rows...Merge:")
    print(f"parquet-tools merge {folder}/{prefix}* {output_name}")
    
    # Code comment
    try:
        import pyarrow.parquet as pq
        import pyarrow as pa
        
        pattern = os.path.join(folder, f"{prefix}*.parquet")
        all_files = sorted(glob.glob(pattern))
        
        writer = None
        total_rows = 0
        
        for file_path in all_files:
            table = pq.read_table(file_path)
            if writer is None:
                writer = pq.ParquetWriter(output_name, table.schema, compression='snappy')
            writer.write_table(table)
            total_rows += table.num_rows
            print(f"Write {os.path.basename(file_path)}: {table.num_rows:,}  rows")
        
        if writer:
            writer.close()
        
        print(f"\n✅ Merge complete... rows...: {total_rows:,}")
        
    except ImportError:
        print("... pyarrow: pip install pyarrow")

##################################################################################################
# Code comment
##################################################################################################

if __name__ == "__main__":
    
    # Code comment
    input_folder = "batch_results_2019_2022"  # Code comment
    
    print("="*60)
    print("Start merging connected ......")
    print("="*60)
    
    # Code comment
    merge_all_parts_efficient(
        prefix="connected_pair_2019_2022_part",             # Comment
        output_name="connected_pair_2019_2022.parquet",     # Comment
        folder=input_folder                                  # Comment
    )
    
    print("\n" + "="*60)
    print("Start merging unconnected ......")
    print("="*60)
    
      # Comment
    merge_all_parts_efficient(
        prefix="unconnected_pair_2019_2022_part",           # Comment
        output_name="unconnected_pair_2019_2022.parquet",   # Comment
        folder=input_folder                                  # Comment
    )
    
    print("\n" + "="*60)
    print("✅ ...Merge...Done...")
    print("="*60)
    
      # Comment
    for fname in ["connected_pair_2019_2022.parquet", "unconnected_pair_2019_2022.parquet"]:
        if os.path.exists(fname):
            size = os.path.getsize(fname) / (1024**3)
            rows = len(pd.read_parquet(fname))  # Code comment
            print(f"{fname}: {rows:,}  rows, {size:.2f} GB")

In [ ]:
import pandas as pd

# Code comment
connected_pair_2019_2022 = pd.read_parquet("connected_pair_2019_2022.parquet")
unconnected_pair_2019_2022 = pd.read_parquet("unconnected_pair_2019_2022.parquet")

  # Comment
print(f"connected_pair_2019_2022: {len(connected_pair_2019_2022)}, unconnected_pair_2019_2022: {len(unconnected_pair_2019_2022)}")


#### unconnected pair in 2022 (no future eval)

In [ ]:
# time_start=time.time()

# unconnected_pairs_2022 = pd.merge(all_combine_pairs_2022, pairs_2022, on=['v1', 'v2'], how='outer', indicator=True)
# unconnected_pairs_2022 = unconnected_pairs_2022[unconnected_pairs_2022['_merge'] == 'left_only']
# unconnected_pairs_2022 = unconnected_pairs_2022.drop(columns=['_merge'])

# print(f"unconnected_pairs_2022: {len(unconnected_pairs_2022)}; elapsed_time: {time.time()-time_start}")


# store_folder="data_pair_solution"
# if not os.path.exists(store_folder):
#     os.makedirs(store_folder)
# print(f"store files in {store_folder}.....")

# ### unconnected pair are connected in 2019, 2022

# time_start = time.time()
# store_name=os.path.join(store_folder,"unconnected_pair_2022.parquet")
# unconnected_pairs_2022.to_parquet(store_name, compression='gzip')
# print(f"unconnected_pairs_2022: {len(unconnected_pairs_2022)}; elapsed_time: {time.time() - time_start}")


In [ ]:
##################################################################################################
# Code comment
##################################################################################################

BATCH_SIZE = 2_000_000  # Code comment
store_folder = "data_pair_solution"
os.makedirs(store_folder, exist_ok=True)
output_folder = os.path.join(store_folder, "unconnected_pairs_2022_parts")
os.makedirs(output_folder, exist_ok=True)

time_start = time.time()
total_rows = len(all_combine_pairs_2022)
num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\n... {total_rows}  rows... {num_batches} ...Process...Generate unconnected_pairs_2022...\n")

for i in range(num_batches):
    batch_start = i * BATCH_SIZE
    batch_end = min((i + 1) * BATCH_SIZE, total_rows)
    print(f"▶ ...Process... {i + 1}/{num_batches} ...:  rows {batch_start} - {batch_end}")

    # Code comment
    batch_df = all_combine_pairs_2022.iloc[batch_start:batch_end].copy()

    # Code comment
    batch_unconnected = pd.merge(batch_df, pairs_2022, on=['v1', 'v2'], how='outer', indicator=True)
    batch_unconnected = batch_unconnected[batch_unconnected['_merge'] == 'left_only'].drop(columns=['_merge'])

      # Comment
    part_path = os.path.join(output_folder, f"unconnected_pairs_2022_part{i+1}.parquet")
    batch_unconnected.to_parquet(part_path, compression="snappy")

    print(f"  ...Save... {i + 1} ... {len(batch_unconnected)}  rows")

      # Comment
    del batch_df, batch_unconnected
    gc.collect()

print(f"\n✅ ...Processing complete...Total time {time.time() - time_start:.2f} s")

##################################################################################################
  # Comment
##################################################################################################

def merge_all_parts(prefix, output_name, folder):
    print(f"\n...Merge... {prefix} ......")
    all_files = sorted([f for f in os.listdir(folder) if f.startswith(prefix)])
    dfs = []
    for f in all_files:
        df = pd.read_parquet(os.path.join(folder, f))
        dfs.append(df)
    merged = pd.concat(dfs, ignore_index=True)
    merged.to_parquet(output_name, compression="gzip")
    print(f"✅ ...MergeSave as {output_name}... rows...: {len(merged)}")
    del dfs, merged
    gc.collect()

# Code comment
# final_store_name = os.path.join(store_folder, "unconnected_pair_2022.parquet")
# merge_all_parts("unconnected_pairs_2022_part", final_store_name, output_folder)


In [ ]:
import pandas as pd
import os
import glob
import gc
import time

def merge_unconnected_parts():
    """
    ...data_pair_solution...unconnected_pairs_2022_partX.parquet...
    ...Merge... itemsunconnected_pair_2022.parquet...
    """
    print("="*60)
    print("Start merging unconnected_pairs_2022 ...")
    print("="*60)
    
    # Code comment
    folder_path = "./data/data_pair_solution"
    parts_folder = os.path.join(folder_path, "unconnected_pairs_2022_parts")
    output_file = os.path.join(folder_path, "unconnected_pair_2022.parquet")
    
      # Comment
    if not os.path.exists(parts_folder):
        print(f"❌ Error: ...does not exist - {parts_folder}")
        return
    
    # Code comment
    pattern = os.path.join(parts_folder, "unconnected_pairs_2022_part*.parquet")
    all_files = sorted(glob.glob(pattern))
    
    if not all_files:
        print(f"❌ Error: ...Found...")
        return
    
    print(f"Found {len(all_files)}  items...")
    
    # Code comment
    if os.path.exists(output_file):
        os.remove(output_file)
        print(f"...Delete...Output...")
    
    # MergeParameters
    BATCH_SIZE = 10  # Code comment
    total_batches = (len(all_files) + BATCH_SIZE - 1) // BATCH_SIZE
    total_rows = 0
    start_time = time.time()
    
    print(f"\nStart...Merge... {total_batches}  items......\n")
    
    # Code comment
    for batch_idx in range(total_batches):
        batch_start = batch_idx * BATCH_SIZE
        batch_end = min((batch_idx + 1) * BATCH_SIZE, len(all_files))
        batch_files = all_files[batch_start:batch_end]
        
        print(f"▶ Process... {batch_idx + 1}/{total_batches} ... ({len(batch_files)}  files)")
        
          # Comment
        batch_dfs = []
        batch_rows = 0
        
        for file_path in batch_files:
            df = pd.read_parquet(file_path)
            file_name = os.path.basename(file_path)
            file_rows = len(df)
            batch_dfs.append(df)
            batch_rows += file_rows
            print(f"  Read {file_name}: {file_rows:,}  rows")
        
          # Comment
        print(f"  Merge... ({batch_rows:,}  rows)...")
        batch_merged = pd.concat(batch_dfs, ignore_index=True)
        
        if batch_idx == 0:
            # Code comment
            batch_merged.to_parquet(output_file, compression='snappy', index=False)
            print(f"  Create...Write {len(batch_merged):,}  rows")
        else:
            # Code comment
            existing_df = pd.read_parquet(output_file)
            combined_df = pd.concat([existing_df, batch_merged], ignore_index=True)
            os.remove(output_file)
            combined_df.to_parquet(output_file, compression='snappy', index=False)
            print(f"  ... {len(batch_merged):,}  rows...Cumulative {len(combined_df):,}  rows")
            del existing_df, combined_df
        
        total_rows += len(batch_merged)
        print(f"  ...Cumulative... rows...: {total_rows:,}  rows\n")
        
          # Comment
        del batch_dfs, batch_merged
        gc.collect()
    
      # Comment
    elapsed_time = time.time() - start_time
    file_size = os.path.getsize(output_file) / (1024**3)
    
    print("="*60)
    print(f"✅ Merge complete...")
    print(f"  Output...: {output_file}")
    print(f"  ... rows...: {total_rows:,}  rows")
    print(f"  ...Size: {file_size:.2f} GB")
    print(f"  Total time: {elapsed_time:.2f} s")
    print("="*60)

# Code comment
if __name__ == "__main__":
    merge_unconnected_parts()

### unconnected pair and solution (citation information); train
(6.3)Merge2019 [description] 
• (1.85)Addcitation=0 columns

In [ ]:
time_start=time.time()
pair_solution_connected_2019=pd.merge(connected_pair_2016_2019,full_graph_2019, on=['v1', 'v2'], how='inner')
print(f"2016 connected in 2019  : {len(pair_solution_connected_2019)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
pair_solution_unconnected_2019=unconnected_pair_2016_2019
pair_solution_unconnected_2019.insert(2, 'citation', 0)
print(f"2016 unconnected in 2019: {len(pair_solution_unconnected_2019)}; elapsed_time: {time.time()-time_start}")

Process2019→2022
◦ 1.9
◦ 1.93

In [ ]:
time_start=time.time()
pair_solution_connected_2022=pd.merge(connected_pair_2019_2022,full_graph_2022, on=['v1', 'v2'], how='inner')
print(f"2019 connected in 2022  : {len(pair_solution_connected_2022)}; elapsed_time: {time.time()-time_start}")

time_start=time.time()
pair_solution_unconnected_2022=unconnected_pair_2019_2022
pair_solution_unconnected_2022.insert(2, 'citation', 0)
print(f"2019 unconnected in 2022: {len(pair_solution_unconnected_2022)}; elapsed_time: {time.time()-time_start}")


#### store orginal cases
 rows
◦ Calculate [description] 
◦ Calculate(citation_m = citation/)
◦ SaveAfter processing

In [ ]:
store_folder="data_pair_solution"
if not os.path.exists(store_folder):
    os.makedirs(store_folder)
print(f"store files in {store_folder}.....")

### unconnected pair are connected in 2019, 2022

time_start = time.time()
store_name=os.path.join(store_folder,"unconnected_2016_pair_solution_connected_2019_full.parquet")
pair_solution_connected_2019.to_parquet(store_name, compression='gzip')
print(f"pair_solution_connected_2019 full: {len(pair_solution_connected_2019)}; elapsed_time: {time.time() - time_start}")


time_start = time.time()
store_name=os.path.join(store_folder,"unconnected_2019_pair_solution_connected_2022_full.parquet")
pair_solution_connected_2022.to_parquet(store_name, compression='gzip')
print(f"pair_solution_connected_2022 full: {len(pair_solution_connected_2022)}; elapsed_time: {time.time() - time_start}\n")


### unconnected pair are not connected in 2019, 2022

time_start = time.time()
store_name=os.path.join(store_folder,"unconnected_2016_pair_solution_unconnected_2019.parquet")
pair_solution_unconnected_2019.to_parquet(store_name, compression='gzip')
print(f"pair_solution_unconnected_2019: {len(pair_solution_unconnected_2019)}; elapsed_time: {time.time() - time_start}")


time_start = time.time()
store_name=os.path.join(store_folder,"unconnected_2019_pair_solution_unconnected_2022.parquet")
pair_solution_unconnected_2022.to_parquet(store_name, compression='gzip')
print(f"pair_solution_unconnected_2022: {len(pair_solution_unconnected_2022)}; elapsed_time: {time.time() - time_start}")

#### merge repeated pairs 

In [ ]:
time_start = time.time()

# Use .groupby to group by 'v1' and 'v2', then use .sum to get the total citations for each pair
grouped_data_df=pair_solution_connected_2019.copy()
grouped_data_df['citation']=pair_solution_connected_2019[['c2019', 'c2018', 'c2017']].sum(axis=1)
dynamic_grouped_data = grouped_data_df.groupby(['v1','v2']).agg({'citation':'sum','v1':'size'}).rename(columns={'v1':'num'}).reset_index()
dynamic_grouped_data['citation_m'] = dynamic_grouped_data[f'citation'] / dynamic_grouped_data['num']
print(f"elapsed_time: {time.time() - time_start}")

time_start = time.time()
store_folder="data_pair_solution" 
store_name=os.path.join(store_folder,"unconnected_2016_pair_solution_connected_2019.parquet")
dynamic_grouped_data.to_parquet(store_name, compression='gzip')
print(f"unconnected_2016_pair_solution_connected_2019: {len(dynamic_grouped_data)}; elapsed_time: {time.time() - time_start}")

In [ ]:
time_start = time.time()
grouped_data_df=pair_solution_connected_2022.copy()
grouped_data_df['citation']=pair_solution_connected_2022[['c2022', 'c2021', 'c2020']].sum(axis=1)
dynamic_grouped_data = grouped_data_df.groupby(['v1','v2']).agg({'citation':'sum','v1':'size'}).rename(columns={'v1':'num'}).reset_index()
dynamic_grouped_data['citation_m'] = dynamic_grouped_data['citation'] / dynamic_grouped_data['num']
print(f"elapsed_time: {time.time() - time_start}")

time_start = time.time()
store_folder="data_pair_solution" 
store_name=os.path.join(store_folder,"unconnected_2019_pair_solution_connected_2022.parquet")
dynamic_grouped_data.to_parquet(store_name, compression='gzip')
print(f"unconnected_2019_pair_solution_connected_2022_processed: {len(dynamic_grouped_data)}; elapsed_time: {time.time() - time_start}")